In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gandhi.a4aman@gmail.com/consolidated_pipeline/src/utils/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar098/{data_source}/*.csv"
print(base_path)

In [0]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', True) \
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')


"""CDF (Change Data Feed) is a Delta Lake feature that tracks row-level changes (inserts, updates, deletes) in a Delta table, simplifying CDC (Change Data Capture) for efficient ETL/ELT, materialized views, and audit trails by recording what changed, how, and when, without scanning entire tables"""

### Silver Processing


In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_duplicate = df_bronze.groupBy('customer_id').count().filter(F.col("count") > 1)
df_duplicate.show()

In [0]:
print("Rows count before deleting duplicates: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print("Rows count after deleting duplicates: ", df_silver.count())


In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name", F.trim(F.col("customer_name"))
)

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
# typos --> correct names

city_mapping = {
    "Bengalore": "Bengaluru",
    "Bengaluruu": "Bengaluru",

    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",

    "NewDelhee": "New Delhi",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi"
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
        .replace(city_mapping, subset=["city"])
        .withColumn(
            "city",
            F.when(F.col("city").isNull(), None)
             .when(F.col("city").isin(allowed), F.col("city"))
             .otherwise(None)
        )
)

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
# Title case fix

df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)


In [0]:
# Business confirmation Note: City corrections confirmed by business team

customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primeful Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad" 
}

# creating a df for fixed_city with customer_id as primary key
df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ['customer_id', 'fixed_city']
)

display(df_fix)

In [0]:
# replacing the null cities with fixed_city by performing LEFT join.

df_silver = (
    df_silver
        .join(df_fix, on="customer_id", how="left")
        .withColumn(
            "city",
            F.coalesce("city", "fixed_city")    # null city will be replaced by fixed_city 
        )
        .drop("fixed_city")
)

df_silver.display()

In [0]:
# sanity check
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)


In [0]:
df_silver = df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)

df_silver.printSchema()

In [0]:
df_silver = (
    df_silver
    # Build final customer col: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        # concat_ws: here ws stands for whitespace
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )

    # Static attributes aligned with parent data
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


### Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")

# take required cols only
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")


In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source = df_child_customers.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()